# Planner Agent
This LangGraph agent reads the dependency report, scans the local codebase, and generates a structured `migration_plan.md`.

In [1]:
import os
import json
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

load_dotenv()

if "OPENAI_API_KEY" not in os.environ:
    from getpass import getpass
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

llm = ChatOpenAI(model="gpt-4o", temperature=0)


### Agent Tools

In [2]:
@tool
def read_migration_report() -> str:
    """Reads the migration_report.json file strictly to gather the details of what dependencies changed. 
    Use this to identify which dependencies required version updates and structural changes (like javax to jakarta).
    """
    try:
        with open("migration_report.json", "r") as f:
            return f.read()
    except FileNotFoundError:
        return "Error: migration_report.json not found."

@tool
def scan_java_codebase(keyword: str) -> str:
    """Scans all .java, .xml, .properties, and .yml files in the spring-petclinic-1.5.x directory for a specific keyword or import.
    Returns a list of file paths that contain the keyword.
    Use this to find where deprecated packages (like 'javax.') are used in the codebase.
    """
    found_files = []
    target_dir = "spring-petclinic-1.5.x"
    extensions = (".java", ".xml", ".properties", ".yml")
    
    try:
        for root, _, files in os.walk(target_dir):
            for file in files:
                if file.endswith(extensions):
                    filepath = os.path.join(root, file)
                    try:
                        with open(filepath, "r", encoding="utf-8") as f:
                            content = f.read()
                            if keyword in content:
                                found_files.append(filepath)
                    except Exception:
                        pass
        
        if not found_files:
            return f"No usages found for '{keyword}'."
        return f"Found '{keyword}' in the following files:\n" + "\n".join(found_files)
    except Exception as e:
        return f"Error scanning codebase: {e}"

tools = [read_migration_report, scan_java_codebase]


### Agent Logic Definition

In [3]:
system_prompt = """You are an Expert Java Migration Planner.

Your goal is to devise a comprehensive, module-by-module migration plan for an old Spring Boot 1.5.x codebase migrating to JDK 17 and Spring Boot 3.x.

Instructions:
1. Always start by reading the dependency migration report using `read_migration_report`.
2. Analyze the dependencies that need updating. Pay special attention to J2EE/Java EE dependencies (like `javax.persistence`, `javax.servlet`, `javax.validation`, `javax.cache` etc.) because upgrading to Spring Boot 3 requires changing `javax.*` imports to `jakarta.*`.
3. Use the `scan_java_codebase` tool to search the codebase for usages of old namespaces (e.g., search for 'javax.persistence', 'javax.validation', 'javax.cache', 'org.hibernate', 'ehcache').
4. Formulate a step-by-step markdown plan describing EXACTLY what files need code changes, which dependencies need replacement in the pom.xml, and any specific refactoring needed.

Your final output must be formatted strictly as a rich Markdown report with actionable steps. Do NOT wrap it in a JSON block.
"""

agent = create_react_agent(llm, tools, prompt=system_prompt)


### Execution

In [4]:
query = "Create a detailed step-by-step migration plan based on the dependency report. Find the files that need updating and output the full markdown plan."

messages = agent.invoke({"messages": [("user", query)]})
final_output = messages["messages"][-1].content

with open("migration_plan.md", "w") as f:
    f.write(final_output)

print("Planner Agent finished! The plan has been saved to migration_plan.md.")

from IPython.display import Markdown
Markdown(final_output)


Planner Agent finished! The plan has been saved to migration_plan.md.


# Migration Plan for Spring Boot 1.5.x to Spring Boot 3.x with JDK 17

This migration plan outlines the necessary steps to upgrade the Spring Boot 1.5.x application to Spring Boot 3.x and JDK 17. The plan includes updating dependencies, refactoring code to accommodate package changes from `javax.*` to `jakarta.*`, and ensuring compatibility with the latest versions.

## Step 1: Update Dependencies in `pom.xml`

1. **Update HSQLDB Dependency**
   - **Current Version**: Managed
   - **Recommended Version**: 2.7.4
   - **Action**: Update the version in `pom.xml`.

2. **Update MySQL Connector Dependency**
   - **Current Version**: Managed
   - **Recommended Version**: 8.0.33
   - **Action**: Update the version in `pom.xml`.

3. **Update Cache API Dependency**
   - **Current Version**: Managed
   - **Recommended Version**: 1.1.1
   - **Action**: Consider removing this dependency and using Spring Boot's built-in caching support.

4. **Update Ehcache Dependency**
   - **Current Version**: Managed
   - **Recommended Version**: 3.10.8
   - **Action**: Update the version in `pom.xml` and ensure to use the 'jakarta' classifier.

5. **Update Webjars Dependencies**
   - **Webjars Locator**: Update to version 0.52
   - **jQuery**: Update to version 3.7.1
   - **jQuery UI**: Update to version 1.14.1
   - **Bootstrap**: Update to version 5.3.7
   - **Action**: Update these versions in `pom.xml`.

## Step 2: Refactor Code for `javax.*` to `jakarta.*`

### Files to Update

1. **`javax.persistence` Usage**
   - **Files**:
     - `src/main/java/org/springframework/samples/petclinic/vet/Vet.java`
     - `src/main/java/org/springframework/samples/petclinic/vet/Specialty.java`
     - `src/main/java/org/springframework/samples/petclinic/owner/Pet.java`
     - `src/main/java/org/springframework/samples/petclinic/owner/Owner.java`
     - `src/main/java/org/springframework/samples/petclinic/owner/PetType.java`
     - `src/main/java/org/springframework/samples/petclinic/visit/Visit.java`
     - `src/main/java/org/springframework/samples/petclinic/model/BaseEntity.java`
     - `src/main/java/org/springframework/samples/petclinic/model/Person.java`
     - `src/main/java/org/springframework/samples/petclinic/model/NamedEntity.java`
   - **Action**: Replace `javax.persistence` imports with `jakarta.persistence`.

2. **`javax.validation` Usage**
   - **Files**:
     - `src/test/java/org/springframework/samples/petclinic/model/ValidatorTests.java`
     - `src/main/java/org/springframework/samples/petclinic/owner/VisitController.java`
     - `src/main/java/org/springframework/samples/petclinic/owner/OwnerController.java`
     - `src/main/java/org/springframework/samples/petclinic/owner/PetController.java`
     - `src/main/java/org/springframework/samples/petclinic/owner/Owner.java`
   - **Action**: Replace `javax.validation` imports with `jakarta.validation`.

3. **`javax.cache` Usage**
   - **Files**:
     - `pom.xml`
     - `src/main/java/org/springframework/samples/petclinic/system/CacheConfig.java`
   - **Action**: Replace `javax.cache` imports with `jakarta.cache` or refactor to use Spring Boot's caching.

4. **`org.hibernate` Usage**
   - **Files**:
     - `src/main/java/org/springframework/samples/petclinic/owner/Owner.java`
     - `src/main/java/org/springframework/samples/petclinic/visit/Visit.java`
     - `src/main/java/org/springframework/samples/petclinic/model/Person.java`
   - **Action**: Ensure compatibility with Hibernate 6.x, which is compatible with Spring Boot 3.x.

5. **`ehcache` Usage**
   - **Files**:
     - `pom.xml`
   - **Action**: Ensure the use of the 'jakarta' classifier for Ehcache.

## Step 3: Testing and Validation

- **Unit Tests**: Run all existing unit tests to ensure they pass with the new configurations.
- **Integration Tests**: Conduct integration testing to verify that all components interact correctly.
- **Manual Testing**: Perform manual testing of critical application paths to ensure functionality.

## Step 4: Deployment

- **Staging Environment**: Deploy the updated application to a staging environment for further testing.
- **Production Deployment**: Once validated, deploy the application to the production environment.

By following this plan, the migration to Spring Boot 3.x and JDK 17 should be smooth, ensuring that the application remains functional and takes advantage of the latest features and improvements.